# 06 - Data Validation
**Goal:** walk through the `DataValidation` component of the pipeline ([data_validation.py](../app/components/data_validation.py)) step by step.

What the component does:
1. **Schema check:** train and test must have the same number of columns as `data_schema/schema.yaml`.
2. **Drift check (Evidently):** compare train (reference) with test (current), column by column.
3. **Route the files:** `validated/` if both checks pass, `invalid/` if not. Reports go to `drift_report/`.

> Input: `train.csv` / `test.csv` from notebook 01 (the same files the ingestion stage produces). Outputs go to `Artifacts/val_nb_*` (gitignored).

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
os.chdir(ROOT)                      # the component reads data_schema/schema.yaml relative to the project root
sys.path.insert(0, str(ROOT))

from app.components.data_validation import DataValidation
from app.entity.config_entity import DataValidationConfig, TrainingPipelineConfig
from app.entity.artifact_entity import DataIngestionArtifact
from app.constants import training_pipeline as tp
from app.utils.main_utils.utils import read_yaml_file

sns.set_theme(style="whitegrid")
SRC = ROOT / "Artifacts" / "notebooks"
train = pd.read_csv(SRC / "train.csv")
test = pd.read_csv(SRC / "test.csv")

def make_validator(run: str, train_path: Path, test_path: Path):
    """Build the component exactly as main.py does, writing into Artifacts/<run>/"""
    cfg = DataValidationConfig(training_pipeline_config=TrainingPipelineConfig(timestamp=run))
    ingestion = DataIngestionArtifact(trained_file_path=train_path, test_file_path=test_path)
    return DataValidation(ingestion, cfg), cfg

print("train:", train.shape, "| test:", test.shape)

## 1. Schema check
The schema file lists every expected column and its type. The component compares only the **number** of columns.

In [ ]:
schema = read_yaml_file(tp.SCHEMA_FILE_PATH)
schema_cols = {k: v for d in schema["columns"] for k, v in d.items()}
print("schema keys:", list(schema))
print("expected columns:", len(schema_cols))
print("numerical columns in schema:", len(schema["numerical_columns"]))

> **Reading the output:** TBD

In [ ]:
validator, cfg = make_validator("val_nb_clean", SRC / "train.csv", SRC / "test.csv")
print("train passes:", validator.validate_num_columns(train))
print("test passes: ", validator.validate_num_columns(test))
print("test minus one column passes:", validator.validate_num_columns(test.drop(columns=["Iframe"])))

> **Reading the output:** TBD

### A stricter check (idea, not in the component yet)
A column **count** can pass even if a column is renamed or has the wrong type. Names and dtypes are cheap to check too.

In [ ]:
def schema_report(df: pd.DataFrame) -> dict:
    return {
        "missing": sorted(set(schema_cols) - set(df.columns)),
        "extra": sorted(set(df.columns) - set(schema_cols)),
        "wrong_dtype": sorted(c for c in df.columns if c in schema_cols and str(df[c].dtype) != schema_cols[c]),
    }

print("clean test:      ", schema_report(test))
renamed = test.rename(columns={"Iframe": "IFrame"})
print("renamed column:  ", schema_report(renamed), "| count check passes:", validator.validate_num_columns(renamed))

> **Reading the output:** TBD

## 2. Drift check on clean data
Train and test come from the same random split, so we expect **no drift**. This is also our baseline for how big the distance is by chance.

In [ ]:
drift = validator.detect_data_drift(base_df=train, current_df=test)
print("drift detected:", drift)

report = read_yaml_file(cfg.drift_report_file_path)
print("summary:", report["summary"])
dist_clean = pd.Series({c: r["distance"] for c, r in report["columns"].items()}).sort_values(ascending=False)
dist_clean.head(6).round(4)

> **Reading the output:** TBD

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
dist_clean.sort_values().plot(kind="barh", ax=ax)
ax.axvline(tp.DATA_DRIFT_THRESHOLD, color="red", ls="--", label=f"threshold {tp.DATA_DRIFT_THRESHOLD}")
ax.set_xlabel("Jensen-Shannon distance"); ax.set_title("Train vs test: no drift"); ax.legend()
plt.show()
print("files written:", sorted(p.name for p in Path(cfg.drift_report_file_path).parent.iterdir()))

> **Reading the output:** TBD

## 3. Drift check on drifted data
Simulate a real change: in 60% of test rows, force `SSLfinal_State` and `URL_of_Anchor` (the two features the model depends on) to `-1`.

In [ ]:
rng = np.random.RandomState(0)
drifted_test = test.copy()
hit = rng.rand(len(drifted_test)) < 0.6
drifted_test.loc[hit, ["SSLfinal_State", "URL_of_Anchor"]] = -1
drifted_test.to_csv(SRC / "test_drifted.csv", index=False)

v2, cfg2 = make_validator("val_nb_drift", SRC / "train.csv", SRC / "test_drifted.csv")
print("drift detected:", v2.detect_data_drift(base_df=train, current_df=drifted_test))
rep2 = read_yaml_file(cfg2.drift_report_file_path)
print({k: v for k, v in rep2["summary"].items() if k != "method"})
dist_drift = pd.Series({c: r["distance"] for c, r in rep2["columns"].items()}).sort_values(ascending=False)
dist_drift.head(4).round(3)

> **Reading the output:** TBD

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
colors = ["tab:red" if d >= tp.DATA_DRIFT_THRESHOLD else "tab:blue" for d in dist_drift.sort_values()]
dist_drift.sort_values().plot(kind="barh", ax=ax, color=colors)
ax.axvline(tp.DATA_DRIFT_THRESHOLD, color="red", ls="--")
ax.set_xlabel("Jensen-Shannon distance"); ax.set_title("Train vs drifted test"); plt.show()

> **Reading the output:** TBD

## 4. Why the decision rule has two parts
How many columns drift, and what does the share rule say on its own?

In [ ]:
share = len(rep2["summary"]["drifted_columns"]) / len(rep2["columns"])
print(f"drifted columns: {rep2['summary']['drifted_columns']}")
print(f"share = {share:.2%} | share rule alone (>= {tp.DATA_DRIFT_SHARE:.0%}) would say drift: {share >= tp.DATA_DRIFT_SHARE}")
print(f"critical columns drifted: {rep2['summary']['drifted_critical_columns']} -> final verdict: {rep2['summary']['drift_detected']}")

> **Reading the output:** TBD

## 5. How big must the shift be? Drift-size sweep
Force a growing fraction of test rows to `-1` for `SSLfinal_State` only, and watch its distance cross the threshold.

In [ ]:
rows = []
for frac in [0.02, 0.05, 0.10, 0.20, 0.40, 0.60]:
    d = test.copy()
    d.loc[np.random.RandomState(1).rand(len(d)) < frac, "SSLfinal_State"] = -1
    p = SRC / "test_sweep.csv"; d.to_csv(p, index=False)
    v, c = make_validator(f"val_nb_sweep_{int(frac*100)}", SRC / "train.csv", p)
    flag = v.detect_data_drift(base_df=train, current_df=d)
    r = read_yaml_file(c.drift_report_file_path)["columns"]["SSLfinal_State"]
    rows.append({"rows_changed": frac, "distance": r["distance"], "column_drifted": r["drift_status"], "dataset_drift": flag})
sweep = pd.DataFrame(rows).set_index("rows_changed")
sweep.round(4)

> **Reading the output:** TBD

## 6. Run the full component
This is the call `main.py` makes. It checks the schema, runs drift, saves the files and returns a `DataValidationArtifact`.

In [ ]:
validator, cfg = make_validator("val_nb_full", SRC / "train.csv", SRC / "test.csv")
artifact = validator.initiate_data_validation()
for k, v in artifact.model_dump().items():
    print(f"{k:<28} {v}")

> **Reading the output:** TBD

In [ ]:
valid_dir = Path(artifact.valid_train_file_path).parent
print("saved:", sorted(p.name for p in valid_dir.iterdir()))
print("train rows preserved:", len(pd.read_csv(artifact.valid_train_file_path)) == len(train),
      "| test rows preserved:", len(pd.read_csv(artifact.valid_test_file_path)) == len(test))

> **Reading the output:** TBD

## 7. Failure path
A test file missing a column. The files must go to `invalid/` and the artifact must say `validation_status=False`.

In [ ]:
bad = test.drop(columns=["Iframe"]); bad.to_csv(SRC / "test_bad.csv", index=False)
vb, cb = make_validator("val_nb_bad", SRC / "train.csv", SRC / "test_bad.csv")
art_bad = vb.initiate_data_validation()
print("validation_status:", art_bad.validation_status)
print("valid paths:      ", art_bad.valid_train_file_path, art_bad.valid_test_file_path)
print("invalid train path:", art_bad.invalid_train_file_path)

> **Reading the output:** TBD

In [ ]:
import contextlib
for p in ["test_drifted.csv", "test_sweep.csv", "test_bad.csv"]:
    with contextlib.suppress(FileNotFoundError):
        (SRC / p).unlink()          # remove the temporary files made in this notebook

## Takeaways
TBD